# QAOA Ising Depth Sweep -- Colab Driver

Thin driver notebook: installs pinned dependencies, clones this repo (on
Colab), and calls straight into `src/experiment.py`'s `run_depth_sweep`. No
sweep or plotting logic is duplicated here -- see `src/experiment.py` for
the actual implementation and `notebooks/analysis.ipynb` for the plots,
which loads the results this notebook saves under `results/` (no Colab
dependency, doesn't recompute anything).

This notebook also runs when opened locally (outside Colab) against an
already-checked-out repo with `requirements.txt` installed -- the setup
cell below detects the environment and skips the pip-install/clone step
in that case.

## Setup

**On Colab**: for the GPU device path, select **Runtime -> Change runtime
type -> GPU** first, then set `COLAB_DEVICE = "GPU"` in the cell below
before running it. This cell always uninstalls both `qiskit-aer` variants
before installing the one you asked for, so it's safe to re-run any time.

In [1]:
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/raumsie/qaoa-ising.git"
COLAB_DEVICE = "GPU"  # "GPU" for the CUDA-enabled qiskit-aer-gpu build (needs a Colab GPU runtime)

if IN_COLAB:
    # Uninstall both variants first
    !pip uninstall -y -q qiskit-aer qiskit-aer-gpu
    if COLAB_DEVICE == "GPU":
        # qiskit-aer-gpu has no release compatible with qiskit 2.x, so this
        # branch pins qiskit==1.1.0 instead of 2.5.0 (recorded in the saved
        # results JSON).
        !pip install -q qiskit==1.1.0 qiskit-aer-gpu==0.15.1 qiskit-algorithms==0.4.0 scipy==1.18.0 matplotlib==3.11.1
    else:
        !pip install -q qiskit==2.5.0 qiskit-aer==0.17.2 qiskit-algorithms==0.4.0 scipy==1.18.0 matplotlib==3.11.1

    import qiskit_aer  # fails fast, here, if the install above didn't actually take
    print(f"qiskit_aer {qiskit_aer.__version__} ready (COLAB_DEVICE={COLAB_DEVICE})")

    # Clone/cd via an absolute path rather than %cd's relative check
    CLONE_PARENT = "/content"
    repo_root = os.path.join(CLONE_PARENT, "qaoa-ising")
    if not os.path.isdir(repo_root):
        os.chdir(CLONE_PARENT)
        !git clone {REPO_URL}
    os.chdir(repo_root)
else:
    # Local run: repo is already checked out and requirements.txt already
    # installed in the current environment --
    # just make sure `src` can be found whether running
    # from the repo root or from notebooks/.
    def _find_repo_root(start):
        d = os.path.abspath(start)
        for _ in range(5):
            if os.path.isdir(os.path.join(d, "src")) and os.path.isfile(os.path.join(d, "requirements.txt")):
                return d
            d = os.path.dirname(d)
        raise RuntimeError(f"Could not locate qaoa-ising repo root from {start}")

    repo_root = _find_repo_root(os.getcwd())
    print(
        "Not running in Colab -- assuming requirements.txt is already "
        "installed in the current environment; skipping pip install/git clone."
    )

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"IN_COLAB={IN_COLAB}, repo_root=./{os.path.basename(repo_root)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 75.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 74.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 18.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 123.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 MB 15.2 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 MB 7.7 MB/s eta 0:00:00:00:01m00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 75.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 MB 10.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

In [2]:
from src.experiment import run_depth_sweep, records_to_dicts
from src.ising_model import generate_test_instances

## Run the depth sweep

Sweeps every (instance) x (QAOA depth `p`) x (optimizer) combination
via `run_depth_sweep` and records `epsilon(p) = (E_qaoa - E_0)/|E_0|`
against the exact-diagonalization baseline for each point.

The constants below are deliberately small for a **fast first run**
(a few minutes on CPU). Increase them at your leisure.

In [3]:
import json
import time

# MINIMIZE_OPTIONS caps optimizer cost for a fast first run (maxiter for
# COBYLA, maxfun for L-BFGS-B). Set to None for an uncapped run.
P_VALUES = range(1, 4)
N_RESTARTS = 2                                 # full sweep: 5
MINIMIZE_OPTIONS = {"maxiter": 50, "maxfun": 60}  # full sweep: None
OPTIMIZER_METHODS = ("COBYLA", "L-BFGS-B")
DEVICE = COLAB_DEVICE                          # setup cell's install choice
SEED = 104
N_SPINS = 6                                    # (qubits) generate_test_instances default

instances = generate_test_instances(n_spins=N_SPINS)
print("Instances:", list(instances.keys()))

t0 = time.time()
records = run_depth_sweep(
    instances=instances,
    p_values=P_VALUES,
    optimizer_methods=OPTIMIZER_METHODS,
    device=DEVICE,
    n_restarts=N_RESTARTS,
    minimize_options=MINIMIZE_OPTIONS,
    seed=SEED,
    verbose=True,
)
elapsed = time.time() - t0
print(f"\nSweep complete: {len(records)} records in {elapsed:.1f}s (device={DEVICE})")

Instances: ['uniform_FM', 'frustrated', 'with_field', 'frustrated_pbc']


/content/qaoa-ising/src/optimizer.py:234: OptimizeWarning: Unknown solver options: maxfun
  scipy_result = minimize(


[uniform_FM] p=1 COBYLA: best_energy=-2.735815 E_0=-5.000000 epsilon=0.452837 (1.04s, 72 fevals)
[uniform_FM] p=1 L-BFGS-B: best_energy=-2.735815 E_0=-5.000000 epsilon=0.452837 (0.85s, 111 fevals)
[uniform_FM] p=2 COBYLA: best_energy=-3.683520 E_0=-5.000000 epsilon=0.263296 (0.93s, 100 fevals)
[uniform_FM] p=2 L-BFGS-B: best_energy=-2.971155 E_0=-5.000000 epsilon=0.405769 (1.83s, 130 fevals)
[uniform_FM] p=3 COBYLA: best_energy=-3.482607 E_0=-5.000000 epsilon=0.303479 (1.70s, 100 fevals)
[uniform_FM] p=3 L-BFGS-B: best_energy=-3.269645 E_0=-5.000000 epsilon=0.346071 (1.31s, 133 fevals)
[frustrated] p=1 COBYLA: best_energy=-0.441817 E_0=-2.593732 epsilon=0.829660 (0.67s, 81 fevals)
[frustrated] p=1 L-BFGS-B: best_energy=-1.819387 E_0=-2.593732 epsilon=0.298545 (0.41s, 51 fevals)
[frustrated] p=2 COBYLA: best_energy=-0.842056 E_0=-2.593732 epsilon=0.675350 (0.97s, 100 fevals)
[frustrated] p=2 L-BFGS-B: best_energy=-0.929680 E_0=-2.593732 epsilon=0.641567 (1.11s, 135 fevals)
[frustrated] 

## Save results

Saved as JSON (via `records_to_dicts`) under `results/` so
`notebooks/analysis.ipynb` can load and plot them without recomputing
anything (and without any Colab dependency).

In [4]:
import qiskit

RESULTS_DIR = os.path.join(repo_root, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

output = {
    "config": {
        "p_values": list(P_VALUES),
        "n_restarts": N_RESTARTS,
        "optimizer_methods": list(OPTIMIZER_METHODS),
        "minimize_options": MINIMIZE_OPTIONS,
        "device": DEVICE,
        "seed": SEED,
        "n_spins": N_SPINS,
        "instance_names": list(instances.keys()),
        "wall_time_s_total": elapsed,
        "qiskit_version": qiskit.__version__,  # differs by device: 1.1.0 for GPU, 2.5.0 for CPU
    },
    "records": records_to_dicts(records),
}

results_path = os.path.join(RESULTS_DIR, "depth_sweep_results.json")
with open(results_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved {len(records)} records to {os.path.relpath(results_path, repo_root)}")

Saved 24 records to results/depth_sweep_results.json


## Optional: CPU-vs-GPU wall-clock comparison

Runs a small matched subset of the sweep on `device="GPU"` and
`device="CPU_AER"` and saves both timings so `analysis.ipynb` can show a
CPU-vs-GPU bar chart. `CPU_AER` (not the sweep's default `"CPU"`) is used
deliberately here: `"CPU"` is qiskit's own plain reference simulator
(`StatevectorEstimator`), a completely different codebase from Aer's
GPU backend, so comparing it against `"GPU"` would measure two different
simulator engines, not hardware. `CPU_AER` runs the identical Aer
`EstimatorV2` code path as `"GPU"`, just with `AerSimulator(device="CPU")`.

This only works on a Colab GPU runtime with `qiskit-aer-gpu` installed.

**Must run the main sweep before this cell**

In [5]:
GPU_TIMING_P_VALUES = range(1, 4)
GPU_TIMING_N_RESTARTS = 2
GPU_TIMING_MINIMIZE_OPTIONS = {"maxiter": 50, "maxfun": 60}
GPU_TIMING_N_SPINS = N_SPINS                   # edit to size this test independently of the main sweep
GPU_TIMING_INSTANCES = {"uniform_FM": generate_test_instances(n_spins=GPU_TIMING_N_SPINS)["uniform_FM"]}

gpu_results_path = os.path.join(RESULTS_DIR, "gpu_timing_results.json")

try:
    t0 = time.time()
    gpu_records = run_depth_sweep(
        instances=GPU_TIMING_INSTANCES,
        p_values=GPU_TIMING_P_VALUES,
        optimizer_methods=OPTIMIZER_METHODS,
        device="GPU",
        n_restarts=GPU_TIMING_N_RESTARTS,
        minimize_options=GPU_TIMING_MINIMIZE_OPTIONS,
        seed=SEED,
        verbose=True,
    )
    gpu_elapsed = time.time() - t0

    t0 = time.time()
    cpu_records = run_depth_sweep(
        instances=GPU_TIMING_INSTANCES,
        p_values=GPU_TIMING_P_VALUES,
        optimizer_methods=OPTIMIZER_METHODS,
        device="CPU_AER",
        n_restarts=GPU_TIMING_N_RESTARTS,
        minimize_options=GPU_TIMING_MINIMIZE_OPTIONS,
        seed=SEED,
        verbose=True,
    )
    cpu_elapsed = time.time() - t0

    gpu_output = {
        "config": {
            "p_values": list(GPU_TIMING_P_VALUES),
            "n_restarts": GPU_TIMING_N_RESTARTS,
            "minimize_options": GPU_TIMING_MINIMIZE_OPTIONS,
            "optimizer_methods": list(OPTIMIZER_METHODS),
            "instance_names": list(GPU_TIMING_INSTANCES.keys()),
            "n_spins": GPU_TIMING_N_SPINS,
            "seed": SEED,
            "cpu_device": "CPU_AER",
            "qiskit_version": qiskit.__version__,
        },
        "gpu_records": records_to_dicts(gpu_records),
        "cpu_records": records_to_dicts(cpu_records),
        "gpu_wall_time_s_total": gpu_elapsed,
        "cpu_wall_time_s_total": cpu_elapsed,
    }
    with open(gpu_results_path, "w") as f:
        json.dump(gpu_output, f, indent=2)

    print(f"Saved CPU-vs-GPU timing comparison to {os.path.relpath(gpu_results_path, repo_root)}")
    print(f"GPU total: {gpu_elapsed:.1f}s, CPU_AER total: {cpu_elapsed:.1f}s")
except Exception as exc:
    print(
        f"GPU timing comparison skipped ({type(exc).__name__}: {exc}). "
        "This is expected outside a Colab GPU runtime with qiskit-aer-gpu "
        "installed -- not an error in the main CPU sweep above."
    )

[uniform_FM] p=1 COBYLA: best_energy=-2.735815 E_0=-5.000000 epsilon=0.452837 (0.59s, 72 fevals)
[uniform_FM] p=1 L-BFGS-B: best_energy=-2.735815 E_0=-5.000000 epsilon=0.452837 (0.93s, 111 fevals)
[uniform_FM] p=2 COBYLA: best_energy=-3.683520 E_0=-5.000000 epsilon=0.263296 (1.27s, 100 fevals)
[uniform_FM] p=2 L-BFGS-B: best_energy=-2.971155 E_0=-5.000000 epsilon=0.405769 (1.84s, 130 fevals)
[uniform_FM] p=3 COBYLA: best_energy=-3.482607 E_0=-5.000000 epsilon=0.303479 (1.04s, 100 fevals)
[uniform_FM] p=3 L-BFGS-B: best_energy=-3.269645 E_0=-5.000000 epsilon=0.346071 (1.46s, 133 fevals)
[uniform_FM] p=1 COBYLA: best_energy=-2.735815 E_0=-5.000000 epsilon=0.452837 (0.48s, 67 fevals)
[uniform_FM] p=1 L-BFGS-B: best_energy=-2.735815 E_0=-5.000000 epsilon=0.452837 (0.72s, 111 fevals)
[uniform_FM] p=2 COBYLA: best_energy=-3.683870 E_0=-5.000000 epsilon=0.263226 (0.85s, 100 fevals)
[uniform_FM] p=2 L-BFGS-B: best_energy=-2.971155 E_0=-5.000000 epsilon=0.405769 (0.98s, 130 fevals)
[uniform_FM]